# ITRMA4 Research Project — Data Preprocessing
## A Comparative Evaluation of Supervised Machine Learning Models for Urban Traffic Congestion Prediction
**Student:** Warona Maphala  
**Dataset:** Zafar, N. & Ul Haq, I. (2020). *Traffic congestion prediction based on Estimated Time of Arrival.* PLoS ONE, 15(12), e0238200.  
**File:** pone_0238200_s003 (TrafficTwoMonth dataset)

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('Libraries imported successfully.')

---
## Step 2: Load Dataset
The dataset is an Excel file (despite the .csv extension). It contains approximately 317,000 traffic records collected via Google Maps API across Islamabad, Pakistan.

In [ ]:
# Load the dataset
df = pd.read_excel('pone_0238200_s003.csv')  # Excel format despite .csv extension

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()
print('Column names:')
print(df.columns.tolist())

In [ ]:
# Preview first few rows
df.head()

In [ ]:
# Data types overview
print('Data Types:')
print(df.dtypes)

---
## Step 3: Descriptive Overview

In [ ]:
# Statistical summary of numeric columns
df[['Fastest_Route_Distance', 'Fastest_Route_Time']].describe().round(2)

In [ ]:
# Class distribution of the target variable
print('Target variable — Data_prediction (Traffic State):')
print()
label_counts = df['Data_prediction'].value_counts()
label_pct = df['Data_prediction'].value_counts(normalize=True).round(4) * 100
dist_df = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pct})
print(dist_df)

# Plot
plt.figure(figsize=(8, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='Blues_d')
plt.title('Class Distribution of Traffic States', fontsize=13)
plt.xlabel('Traffic State')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

---
## Step 4: Missing Value Check

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print()
print(f'Total missing values: {missing.sum()}')
print('\nConclusion: No missing values found. Dataset is complete.')

---
## Step 5: Data Transformation

### 5.1 Parse Date Column
Convert the `Date` column from string to a proper datetime object and extract temporal features for analysis.

In [ ]:
# Parse date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

print('Date range in dataset:')
print(f'  Start : {df["Date"].min().date()}')
print(f'  End   : {df["Date"].max().date()}')
print(f'  Span  : {(df["Date"].max() - df["Date"].min()).days} days')

### 5.2 Extract Time-of-Day Features
The `Sys_Time` column contains the time of observation. We extract the hour and classify each record as peak-hour or non-peak-hour, consistent with Zafar & Ul Haq (2020), who identified 8–9 am, 2–3 pm, and 6–7 pm as peak periods.

In [ ]:
# Extract hour from Sys_Time
df['Hour'] = pd.to_datetime(df['Sys_Time'].astype(str), format='%H:%M:%S').dt.hour

# Define peak hours consistent with Zafar & Ul Haq (2020)
# Peak: 7–9 am, 12–15 pm, 17–20 pm
peak_hours = list(range(7, 10)) + list(range(12, 16)) + list(range(17, 21))
df['Time'] = df['Hour'].apply(lambda h: 'peak_hour' if h in peak_hours else 'non_peak_hour')

print('Time classification distribution:')
print(df['Time'].value_counts())

### 5.3 Day of Week — Numeric Encoding
Convert `Day` from string to numeric (Monday=0 ... Sunday=6), as per Zafar & Ul Haq (2020).

In [ ]:
day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}
df['Day_Numeric'] = df['Day'].map(day_map)

print('Day mapping applied:')
print(df[['Day', 'Day_Numeric']].drop_duplicates().sort_values('Day_Numeric').to_string(index=False))

---
## Step 6: Feature Extraction — Congestion Index (CI)

Following Zafar & Ul Haq (2020), the Congestion Index is computed per road segment as:

> **CI = (t_L − t_O) / t_O**

Where:
- `t_L` = current travel time for the segment
- `t_O` = minimum (free-flow) travel time for that segment

This normalises travel times across road segments of different lengths.

In [ ]:
# Compute minimum travel time per starting location (segment baseline)
t_min = df.groupby('Starting_Location')['Fastest_Route_Time'].transform('min')

# Compute Congestion Index
df['CI'] = (df['Fastest_Route_Time'] - t_min) / t_min

print('Congestion Index computed successfully.')
print()
print('CI descriptive statistics:')
print(df['CI'].describe().round(4))

In [ ]:
# Verify CI thresholds align with Zafar & Ul Haq (2020) Table 2
# Smooth: CI < 0.15 | Slightly Congested: 0.15–0.35 | Congested: 0.35–0.65
# Highly Congested: 0.65–2.0  | Blockage: > 2.0

bins = [-0.001, 0.15, 0.35, 0.65, 2.0, df['CI'].max() + 1]
labels_ci = ['smooth', 'slightly congested', 'congested', 'highly congested', 'blockage']
df['CI_Label'] = pd.cut(df['CI'], bins=bins, labels=labels_ci)

print('CI-derived label distribution:')
print(df['CI_Label'].value_counts())
print()
print('Original Data_prediction distribution (for comparison):')
print(df['Data_prediction'].value_counts())

---
## Step 7: Encoding Categorical Features

Categorical variables are converted to numeric values. Binary features (`Holiday`, `Special_Condition`, `Time`) are encoded as 0/1. Multi-class features (`Weather`) use LabelEncoder.

In [ ]:
# --- Binary encoding ---
df['Holiday_Enc'] = df['Holiday'].map({'yes': 1, 'no': 0})
df['SpecialCondition_Enc'] = df['Special_Condition'].map({'yes': 1, 'no': 0})
df['Time_Enc'] = df['Time'].map({'peak_hour': 1, 'non_peak_hour': 0})

print('Binary encoding complete.')
print(f"  Holiday — unique values after encoding: {df['Holiday_Enc'].unique()}")
print(f"  Special Condition — unique values: {df['SpecialCondition_Enc'].unique()}")
print(f"  Time — unique values: {df['Time_Enc'].unique()}")

In [ ]:
# --- Label encoding for Weather ---
le_weather = LabelEncoder()
df['Weather_Enc'] = le_weather.fit_transform(df['Weather'])

print('Weather encoding:')
for i, cls in enumerate(le_weather.classes_):
    print(f'  {i} → {cls}')

---
## Step 8: Feature Binarisation (Target Label Encoding)

The target variable `Data_prediction` is encoded numerically as follows, consistent with Zafar & Ul Haq (2020):
- `smooth` → 0  
- `slightly congested` → 1  
- `congested` → 2  
- `highly congested` → 3  
- `blockage` → 4

In [ ]:
label_map = {
    'smooth': 0,
    'slightly congested': 1,
    'congested': 2,
    'highly congested': 3,
    'blockage': 4
}
df['Target'] = df['Data_prediction'].map(label_map)

print('Target variable encoding:')
print(df[['Data_prediction', 'Target']].drop_duplicates().sort_values('Target').to_string(index=False))
print()
print('Target class distribution:')
print(df['Target'].value_counts().sort_index())

---
## Step 9: Feature Selection — Final Model Features

Select the final preprocessed features for use in model training, following the feature set from Zafar & Ul Haq (2020): Day, Weather, Time, Holiday, Special Condition, and the derived CI feature.

In [ ]:
features = [
    'Day_Numeric',          # Day of the week (0–6)
    'Weather_Enc',          # Weather condition (encoded)
    'Time_Enc',             # Peak/non-peak hour (0/1)
    'Holiday_Enc',          # Holiday flag (0/1)
    'SpecialCondition_Enc', # Special condition flag (0/1)
    'CI',                   # Congestion Index (derived)
    'Hour',                 # Hour of day
    'Fastest_Route_Distance' # Road segment distance
]
target = 'Target'

X = df[features]
y = df[target]

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print()
print('Preview of preprocessed feature matrix:')
X.head()

---
## Step 10: Exploratory Visualisations

In [ ]:
# Congestion index distribution by traffic state
plt.figure(figsize=(10, 5))
order = ['smooth', 'slightly congested', 'congested', 'highly congested', 'blockage']
sns.boxplot(data=df, x='Data_prediction', y='CI', order=order, palette='Blues')
plt.title('Congestion Index Distribution by Traffic State', fontsize=13)
plt.xlabel('Traffic State')
plt.ylabel('Congestion Index (CI)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('ci_by_state.png', dpi=150)
plt.show()

In [ ]:
# Average CI by hour of day
hourly_ci = df.groupby('Hour')['CI'].mean()

plt.figure(figsize=(10, 4))
plt.plot(hourly_ci.index, hourly_ci.values, marker='o', color='steelblue', linewidth=2)
plt.fill_between(hourly_ci.index, hourly_ci.values, alpha=0.15, color='steelblue')
plt.title('Average Congestion Index by Hour of Day', fontsize=13)
plt.xlabel('Hour of Day')
plt.ylabel('Mean CI')
plt.xticks(range(0, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ci_by_hour.png', dpi=150)
plt.show()

In [ ]:
# Traffic state distribution across weather conditions
weather_state = df.groupby(['Weather', 'Data_prediction']).size().unstack(fill_value=0)
weather_state_pct = weather_state.div(weather_state.sum(axis=1), axis=0) * 100

weather_state_pct[order].plot(kind='bar', stacked=True, figsize=(11, 5),
    colormap='Blues', edgecolor='white')
plt.title('Traffic State Distribution by Weather Condition', fontsize=13)
plt.xlabel('Weather Condition')
plt.ylabel('Percentage (%)')
plt.legend(loc='lower right', fontsize=8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('state_by_weather.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = X.corr().round(2)
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

---
## Step 11: Export Preprocessed Dataset

In [ ]:
# Save the final preprocessed dataframe
df_preprocessed = df[features + [target, 'Date', 'Day', 'Data_prediction']].copy()
df_preprocessed.to_csv('traffic_preprocessed.csv', index=False)

print(f'Preprocessed dataset saved: traffic_preprocessed.csv')
print(f'Shape: {df_preprocessed.shape}')
print()
print('Final column summary:')
print(df_preprocessed.dtypes)

---
## Summary of Preprocessing Steps

| Step | Action | Result |
|------|--------|--------|
| Load | Read Excel file | 317,112 rows × 12 columns |
| Missing values | Check nulls | None found — dataset complete |
| Date parsing | Convert to datetime | Date range: 13–29 Feb 2020 |
| Time extraction | Hour + peak classification | `Hour`, `Time_Enc` features created |
| Day encoding | String → numeric (0–6) | `Day_Numeric` created |
| CI computation | (t_L − t_O) / t_O per segment | `CI` feature derived |
| Categorical encoding | Weather LabelEncoder; binary 0/1 | `Weather_Enc`, `Holiday_Enc`, `SpecialCondition_Enc` |
| Target encoding | 5-class label → 0–4 | `Target` column |
| Export | CSV saved | `traffic_preprocessed.csv` |